In [ ]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

In [ ]:
product_features = (
    df.groupby("skuNumber")
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        avg_qty=(
            "effective_qty",
            "mean"
        )
    )
    .reset_index()
)

product_features.head()

In [ ]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category",
            "subCategory"
        ]
    ]
    .drop_duplicates()
)

product_features = (
    product_features.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

product_features.head()

In [ ]:
total_retailers = (
    df["customerId"]
    .nunique()
)

product_features[
    "popularity_score"
] = (
    product_features[
        "unique_buyers"
    ]
    / total_retailers
)

product_features.head()

In [ ]:
(
    product_features[
        [
            "itemName",
            "popularity_score"
        ]
    ]
    .sort_values(
        "popularity_score",
        ascending=False
    )
    .head(10)
)

In [ ]:
product_features[
    "category_rank"
] = (
    product_features
    .groupby("category")
    ["total_qty"]
    .rank(
        ascending=False,
        method="dense"
    )
)

product_features.head()

In [ ]:
product_features.to_parquet(
    "../data/processed/product_features.parquet",
    index=False
)

print("Saved")